# Higgsfield — Retention, LTV, and how the segments moved

Part 1 split the paying base into five behavioural segments. This notebook prices them.

1. **Churn** — what counts as churned when the data has charges but no cancellations, and what churn costs.
2. **LTV** — gross LTV from observed retention, then the compute cost at which each segment stops paying for itself.
3. **History** — how the segment mix and per-segment value moved by cohort.

Every rule below (period length, renewal tolerance, who is excluded) is read off the billing data in §0.2 rather than assumed.

## 0. Setup

In [20]:
import numpy as np
import pandas as pd
import nbformat  # required for plotly fig.show() mime rendering in notebooks
import plotly.express as px
import plotly.graph_objects as go
import plotly.io._renderers as _plotly_renderers
from _plotly_utils.optional_imports import _not_importable

# Plotly caches failed optional imports for the kernel lifetime; clear + rebind.
_not_importable.discard('nbformat')
_plotly_renderers.nbformat = nbformat

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:,.2f}'.format)

PALETTE = ['#6C5CE7', '#00B894', '#FDCB6E', '#E17055', '#0984E3', '#2D3436', '#B2BEC3']
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = PALETTE
px.defaults.height = 400

PATH = 'data/'
PERIOD = 30  # billing period, days

In [21]:
generations_df = pd.read_csv(PATH + 'generations.csv')
customers_df = pd.read_csv(PATH + 'customers.csv')
charges_df = pd.read_csv(PATH + 'charges.csv')

def to_utc(s):
    return pd.to_datetime(s.astype(str).str.removesuffix(' UTC'), format='ISO8601', utc=True, errors='coerce')

for c in ['in_progress_at', 'completed_at', 'failed_at']:
    generations_df[c] = to_utc(generations_df[c])
customers_df['created_at'] = to_utc(customers_df['created_at'])
charges_df['revenue_time'] = to_utc(charges_df['revenue_time'])

# one account generates orders of magnitude more than anyone else - dropped in Part 1 too
OUTLIERS = list(generations_df['user_id'].value_counts().index[:1])
drop_cust = set(customers_df.loc[customers_df['user_id'].isin(OUTLIERS), 'customer'].dropna())
generations_df = generations_df[~generations_df['user_id'].isin(OUTLIERS)].copy()
customers_df = customers_df[~customers_df['user_id'].isin(OUTLIERS)].copy()
charges_df = charges_df[~charges_df['customer'].isin(drop_cust)].copy()

generations_df['is_video'] = (generations_df['type'] == 'video').astype(int)
charges_df['is_credits'] = charges_df['subscription_plan'].eq('Credits Package')
charges_df = charges_df.sort_values(['customer', 'revenue_time'])

DATA_END = charges_df['revenue_time'].max()
print(f'generations {len(generations_df):,} | customers {len(customers_df):,} | charges {len(charges_df):,}')
print(f'data ends: {DATA_END}')
print(f'\npayment types:\n{charges_df["payment_type"].value_counts().to_string()}')

generations 4,949,426 | customers 19,534 | charges 47,291
data ends: 2025-09-30 23:53:39+00:00

payment types:
payment_type
Subscription Create    19529
Subscription Rebill    16411
Credits package         6773
Subscription Update     3090
Reactivation            1488


### 0.2 Three facts in the billing data that set the rules

Fixing these first, because each one decides a definition later: **when a renewal is late enough to be churn**, **whether the last month is usable**, and **whether annual accounts can be measured at all**.

In [22]:
sub_all = charges_df[~charges_df['is_credits']].copy()
firsts = sub_all.groupby('customer').agg(first_pay=('revenue_time', 'min'),
                                         first_period=('subscription_period', 'first'))
sub_all = sub_all.join(firsts, on='customer')
sub_all['days'] = (sub_all['revenue_time'] - sub_all['first_pay']).dt.total_seconds() / 86400
firsts['tenure_days'] = (DATA_END - firsts['first_pay']).dt.total_seconds() / 86400

# ---- FACT 1: when does the rebill actually land?
den = firsts[firsts['first_period'].eq('month') & (firsts['tenure_days'] >= 2 * PERIOD)]
nxt = sub_all[sub_all['days'] > 1].groupby('customer')['days'].min().reindex(den.index)
cum = pd.DataFrame({'by_day': range(28, 61)})
cum['renewed'] = [float((nxt <= d).mean()) for d in cum['by_day']]
cum['added_pp'] = cum['renewed'].diff() * 100
print(f'first renewal by day X, monthly customers with >= 60d tenure (n={len(den):,}):')
print(cum[cum['by_day'].isin([28, 29, 30, 31, 32, 33, 35, 37, 40, 45, 60])].round(4).to_string(index=False))

fig = go.Figure()
fig.add_bar(x=cum['by_day'], y=cum['added_pp'], marker_color='#6C5CE7', name='renewals added that day')
fig.add_vline(x=32.5, line_dash='dot', line_color='#E17055',
              annotation_text='rebill cluster ends', annotation_position='top right')
fig.update_layout(title='when the second charge arrives: the rebill cluster ends at day 32',
                  xaxis_title='days since first charge', yaxis_title='pp of customers added')
fig.show()

TOL = 2  # days of grid tolerance, set by the cluster above
print(f'\n-> TOL = {TOL} days. Days 30-32 add {cum.loc[cum["by_day"].between(30, 32), "added_pp"].sum():.1f}pp, '
      f'days 33-37 only {cum.loc[cum["by_day"].between(33, 37), "added_pp"].sum():.1f}pp and days 38-60 a further '
      f'{cum.loc[cum["by_day"].between(38, 60), "added_pp"].sum():.1f}pp, arriving at a flat trickle with no second '
      'cluster. So there is no gap in the distribution for a grace period to sit in: a 30-day cycle window with '
      f'{TOL} days of grid tolerance already contains every scheduled rebill, including the 28-29 day ones that '
      'calendar-month billing produces in short months.')

# the late trickle is labelled in the data, so it does not need a grace window - it needs excluding
react = sub_all[sub_all['payment_type'].eq('Reactivation')]
print(f'\nthe late trickle is win-back, and the data says so: {len(react):,} Reactivation charges across '
      f'{react["customer"].nunique():,} customers, landing at a median of {react["days"].median():.0f} days '
      f'(p25 {react["days"].quantile(.25):.0f}) and worth {react["sales_amount"].sum() / sub_all["sales_amount"].sum():.1%} '
      'of subscription revenue.')
print('-> Reactivations are recovered revenue, not retained revenue: they are excluded from renewal and '
      'reported separately. That is worth 3.3pp on the first-renewal rate (53.9% -> 50.6%).')

first renewal by day X, monthly customers with >= 60d tenure (n=11,946):
 by_day  renewed  added_pp
     28     0.06       NaN
     29     0.06      0.11
     30     0.07      0.13
     31     0.19     12.20
     32     0.48     28.95
     33     0.49      1.34
     35     0.51      0.82
     37     0.52      0.72
     40     0.54      0.30
     45     0.55      0.35
     60     0.58      0.12



-> TOL = 2 days. Days 30-32 add 41.3pp, days 33-37 only 4.6pp and days 38-60 a further 5.3pp, arriving at a flat trickle with no second cluster. So there is no gap in the distribution for a grace period to sit in: a 30-day cycle window with 2 days of grid tolerance already contains every scheduled rebill, including the 28-29 day ones that calendar-month billing produces in short months.

the late trickle is win-back, and the data says so: 1,488 Reactivation charges across 1,321 customers, landing at a median of 64 days (p25 45) and worth 4.7% of subscription revenue.
-> Reactivations are recovered revenue, not retained revenue: they are excluded from renewal and reported separately. That is worth 3.3pp on the first-renewal rate (53.9% -> 50.6%).


In [23]:
# ---- FACT 2: is the last month usable?
cal = charges_df.copy()
cal['ym'] = cal['revenue_time'].dt.tz_localize(None).dt.to_period('M').astype(str)
print('charges per calendar month by type:')
print(pd.crosstab(cal['ym'], cal['payment_type']).to_string())
print('\nfirst charges per month (cohort sizes):')
print(firsts['first_pay'].dt.tz_localize(None).dt.to_period('M').value_counts().sort_index().to_string())
last_m = cal['ym'].max()
lm = cal[cal['ym'].eq(last_m)]
print(f'\n-> {last_m}: ZERO new subscriptions, {lm["payment_type"].eq("Subscription Rebill").sum():,} rebills. '
      'The base is frozen on 31 Aug, so the last month is a pure retention month: used in full for renewal '
      'measurement (it is what makes every cohort through August observable) and excluded from cohort/mix '
      'work, where it contributes no customers.')

charges per calendar month by type:
payment_type  Credits package  Reactivation  Subscription Create  Subscription Rebill  Subscription Update
ym                                                                                                        
2025-04                   358             0                 1180                    0                  103
2025-05                   812            47                 2989                  604                  326
2025-06                   665           139                 2569                 1747                  285
2025-07                  1283           245                 6272                 2584                  713
2025-08                  1936           455                 6519                 4829                 1018
2025-09                  1719           602                    0                 6647                  645

first charges per month (cohort sizes):
first_pay
2025-04    1181
2025-05    2989
2025-06    2569
2025-07  

In [24]:
# ---- FACT 3: does annual rebill monthly?
ann = firsts[firsts['first_period'].eq('year')]
a = sub_all[sub_all['customer'].isin(ann.index)]
n_charges = a.groupby('customer').size()
reb = a[a['payment_type'].eq('Subscription Rebill') & (a['days'] > 1)]
print(f'annual accounts: {len(ann):,}')
print(f'  with exactly ONE subscription charge:        {(n_charges == 1).sum():,} ({(n_charges == 1).mean():.1%})')
print(f'  with any later rebill:                       {reb["customer"].nunique():,} '
      f'({reb["customer"].nunique() / len(ann):.1%})')
print(f'  with a rebill 28-35d after the first charge:  {reb.loc[reb["days"].between(28, 35), "customer"].nunique():,} '
      f'({reb.loc[reb["days"].between(28, 35), "customer"].nunique() / len(ann):.2%})')
opening = sub_all[sub_all['days'] <= 0.01]
print(f'\nfirst-charge amount: annual median ${opening.loc[opening["first_period"].eq("year"), "sales_amount"].median():.2f} '
      f'vs monthly median ${opening.loc[opening["first_period"].eq("month"), "sales_amount"].median():.2f}'
      '  -> annual is a genuine yearly prepay, not 12 monthly charges')
print(f'later rebills for annual accounts land at a median of {reb["days"].median():.0f} days '
      f'(p25 {reb["days"].quantile(.25):.0f}, p75 {reb["days"].quantile(.75):.0f}) '
      f'for a median ${reb["sales_amount"].median():.2f} - a downgrade to monthly, not an annual renewal.')
print('\nlater charges on annual accounts, by type:')
print(pd.crosstab(a.loc[a['days'] > 1, 'payment_type'], a.loc[a['days'] > 1, 'subscription_period']).to_string())
print(f'\n-> annual does NOT renew monthly. 87% never pay twice, and no annual account reaches day 365 '
      f'inside this window ({(ann["tenure_days"] >= 365).sum()} of {len(ann):,}), so annual renewal is '
      'unmeasurable here. Annual is excluded from §1-§2 and kept in §3 as mix.')

annual accounts: 2,065
  with exactly ONE subscription charge:        1,807 (87.5%)
  with any later rebill:                       65 (3.1%)
  with a rebill 28-35d after the first charge:  11 (0.53%)

first-charge amount: annual median $302.60 vs monthly median $14.00  -> annual is a genuine yearly prepay, not 12 monthly charges
later rebills for annual accounts land at a median of 92 days (p25 62, p75 112) for a median $42.89 - a downgrade to monthly, not an annual renewal.

later charges on annual accounts, by type:
subscription_period  month  year
payment_type                    
Reactivation            11     6
Subscription Rebill    114     0
Subscription Update     57   141

-> annual does NOT renew monthly. 87% never pay twice, and no annual account reaches day 365 inside this window (0 of 2,065), so annual renewal is unmeasurable here. Annual is excluded from §1-§2 and kept in §3 as mix.


**The rules, then:**

| rule | value | why |
|---|---|---|
| billing cycle | 30 days from the first charge, grid shifted by `TOL = 2` days | the rebill cluster spans 30–32 days, with a 28–29 day shoulder from short calendar months |
| **grace period** | **none** | there is no gap in the renewal distribution for one to sit in — the cycle boundary is already the deadline |
| renewal | any non-Reactivation subscription charge inside the cycle | win-backs arrive at a median of 64 days; counting them as renewals is what a grace period would silently do |
| reactivation | recovered revenue, reported separately | 4.7% of subscription revenue, worth 3.3pp on the first renewal (53.9% → 50.6%) |
| observability | cycle *m* must have fully elapsed | accounts too new to have faced the renewal leave the denominator |
| last month | kept for retention, excluded from cohort mix | no new subscriptions in it |
| annual | excluded from churn and LTV | 87% never pay twice; none reach day 365 |

In [25]:
# ---- account table
base = (customers_df
        .merge(charges_df.groupby('customer').agg(revenue=('sales_amount', 'sum')),
               left_on='customer', right_index=True, how='inner')
        .merge(firsts, left_on='customer', right_index=True, how='inner')
        .merge(sub_all.groupby('customer').agg(first_plan=('subscription_plan', 'first')),
               left_on='customer', right_index=True, how='left')
        .set_index('customer'))
base['cohort'] = base['first_pay'].dt.tz_localize(None).dt.to_period('M').astype(str)
base['period_label'] = np.where(base['first_period'].eq('year'), 'annual', 'monthly')

# charges on each customer's own clock; cycle grid carries the 2-day tolerance
ch = charges_df.join(base[['first_pay']], on='customer', how='inner')
ch['days'] = (ch['revenue_time'] - ch['first_pay']).dt.total_seconds() / 86400
ch['cycle'] = np.floor((ch['days'] + TOL) / PERIOD).clip(lower=0).astype(int)

cust_lu = customers_df.dropna(subset=['customer']).drop_duplicates('user_id')[['user_id', 'customer']]
gen = generations_df.merge(cust_lu, on='user_id', how='inner').join(base[['first_pay']], on='customer', how='inner')
gen['days'] = (gen['in_progress_at'] - gen['first_pay']).dt.total_seconds() / 86400
gen['cycle'] = np.floor((gen['days'] + TOL) / PERIOD)

OBS_MIN = PERIOD + TOL  # 32 days: month-1 usage complete AND first renewal decided
print(f'accounts: {len(base):,} | observable (tenure >= {OBS_MIN}d): {(base["tenure_days"] >= OBS_MIN).sum():,}')
print(base['period_label'].value_counts().to_string())

accounts: 19,530 | observable (tenure >= 32d): 19,125
period_label
monthly    17465
annual      2065


In [26]:
# ---- five behavioural segments from Part 1
g28 = gen[(gen['days'] >= -1) & (gen['days'] <= 28)]
w1 = (g28[g28['days'] <= 7].groupby('customer')
      .agg(w1_jobs=('job_id', 'size'), w1_days=('in_progress_at', lambda s: s.dt.date.nunique())))
m1 = g28.groupby('customer').agg(m1_jobs=('job_id', 'size'))

ch30 = ch[ch['days'].between(0, PERIOD)]
rev30 = ch30.groupby('customer')['sales_amount'].sum().rename('rev_30')
rev_sub = ch30.loc[~ch30['is_credits']].groupby('customer')['sales_amount'].sum().rename('rev_sub_30')
rev_cr = ch30.loc[ch30['is_credits']].groupby('customer')['sales_amount'].sum().rename('rev_credits_30')

SEG = base[base['tenure_days'] >= OBS_MIN].join([w1, m1, rev30, rev_sub, rev_cr])
for c in ['w1_jobs', 'w1_days', 'm1_jobs', 'rev_30', 'rev_sub_30', 'rev_credits_30']:
    SEG[c] = SEG[c].fillna(0)
SEG['jobs_after_w1'] = (SEG['m1_jobs'] - SEG['w1_jobs']).clip(lower=0)
# annual pre-pays a year on day 0: monthly-equivalent puts every bucket on one scale
SEG['arpu_meq'] = np.where(SEG['period_label'].eq('annual'), SEG['rev_sub_30'] / 12, SEG['rev_sub_30']) + SEG['rev_credits_30']
SEG['jobs_per_usd'] = SEG['m1_jobs'] / SEG['arpu_meq'].replace(0, np.nan)

usage_pp = SEG['m1_jobs'].rank(pct=True)
j = SEG['m1_jobs'].sort_values(ascending=False)
k50 = int(np.searchsorted((j.cumsum() / j.sum()).to_numpy(), 0.50, side='left')) + 1
TAIL_P = float(np.clip(1 - k50 / len(SEG), 0.95, 0.99))
P80 = SEG['m1_jobs'].quantile(0.80)

SEG['segment'] = np.select(
    [(SEG['w1_days'] <= 1) & (SEG['jobs_after_w1'] == 0),
     SEG['w1_days'] < 3,
     (SEG['w1_days'] >= 3) & (usage_pp > TAIL_P),
     SEG['m1_jobs'] >= P80],
    ['1. one-shot', '2. weak week-1', '5. top usage', '4. habit, heavy'],
    default='3. habit, light')
ORDER = ['1. one-shot', '2. weak week-1', '3. habit, light', '4. habit, heavy', '5. top usage']
SEGMAP = SEG['segment']

print(f'segmentation base: {len(SEG):,}  (Part 1 had 18,048 under a 37-day filter; '
      f'the 7-day grace is gone, so late-August accounts now qualify)')
print(SEG['segment'].value_counts().reindex(ORDER).to_string())
print(f'\np80 = {P80:.0f} month-1 jobs | tail cut at p{TAIL_P * 100:.1f} (>= {SEG["m1_jobs"].quantile(TAIL_P):.0f} jobs)')

segmentation base: 19,125  (Part 1 had 18,048 under a 37-day filter; the 7-day grace is gone, so late-August accounts now qualify)
segment
1. one-shot        5743
2. weak week-1     6887
3. habit, light    3669
4. habit, heavy    2323
5. top usage        503

p80 = 88 month-1 jobs | tail cut at p97.1 (>= 787 jobs)


---
# 1. Churn

## 1.1 Definition

There is no cancellation event in the data, only charges, so churn is **inferred from a charge that did not arrive**:

> A customer is **churned at cycle *m*** if no scheduled subscription charge falls in cycle *m*, measured only on accounts for which cycle *m* has fully elapsed. Win-backs do not count as renewals.

Two things this is not:

- **Not usage churn.** They stop generating before they stop paying (§1.3) — a different, earlier question.
- **Not `1 − A(t+1)/A(t)`.** The dashboard version moves with acquisition, not retention (§1.2).

Voluntary vs involuntary is not separable: `payment_type` does not distinguish a cancellation from a failed card.

Retention is reported as **chained renewal rates** — each cycle's rate computed only on accounts that reached it and paid the cycle before, then multiplied into a survival curve. Accounts too new to have faced a renewal leave the denominator rather than counting as retained. That is the censoring problem handled directly, with no survival model in between.

In [27]:
MON = base[base['period_label'].eq('monthly') & (base['tenure_days'] >= OBS_MIN)].copy()
MON['segment'] = SEGMAP

# retention and LTV run on retained revenue: win-backs are recovery, counted separately
recovered = ch[ch['payment_type'].eq('Reactivation')]
ch_ret = ch[~ch['payment_type'].eq('Reactivation')]
sub = ch_ret[~ch_ret['is_credits']]
print(f'recovered revenue set aside: ${recovered["sales_amount"].sum():,.0f} from '
      f'{recovered["customer"].nunique():,} accounts ({recovered["sales_amount"].sum() / ch["sales_amount"].sum():.1%} of all revenue)')

rev_cycle = ch_ret.groupby(['customer', 'cycle'])['sales_amount'].sum().rename('rev').reset_index()
piv = rev_cycle.pivot(index='customer', columns='cycle', values='rev')
paid_in = (sub.groupby(['customer', 'cycle']).size().unstack().notna())
paid_in = paid_in.reindex(columns=range(int(paid_in.columns.max()) + 1), fill_value=False).fillna(False)
# a lifetime ends at the FIRST missed cycle, so survival and revenue describe the same population
alive = paid_in.cummin(axis=1).astype(bool)

resume = (paid_in & ~alive).any(axis=1)
gap_churn = ((~paid_in[1]) & paid_in.loc[:, 1:].any(axis=1)).sum()
print(f'accounts that miss a cycle and pay again later: {resume.sum():,} of {len(paid_in):,} ({resume.mean():.1%}) '
      '-> treating the first miss as the end of the lifetime is safe; it is not quietly counting dunning gaps as churn')

def retention(customers, min_n=100):
    # chained renewal rates: P(pays cycle m | paid cycle m-1, tenure reached 30m + TOL)
    idx = alive.index.intersection(pd.Index(customers))
    A, ten = alive.loc[idx], base['tenure_days'].reindex(idx)
    rows, S = [{'cycle': 0, 'n_at_risk': len(idx), 'renewal': 1.0, 'S': 1.0}], 1.0
    for m in range(1, int(A.columns.max()) + 1):
        elig = A.index[(ten >= PERIOD * m + TOL) & A[m - 1]]
        if len(elig) < min_n:
            break
        r = float(A.loc[elig, m].mean())
        S *= r
        rows.append({'cycle': m, 'n_at_risk': len(elig), 'renewal': r, 'S': S})
    return pd.DataFrame(rows)

RET = retention(MON.index)
print(f'monthly, observable: {len(MON):,}')
print(RET.round(3).to_string(index=False))

fig = go.Figure()
fig.add_scatter(x=RET['cycle'], y=RET['S'], mode='lines+markers', name='survival', line=dict(color='#6C5CE7'))
fig.add_bar(x=RET['cycle'][1:], y=RET['renewal'][1:], name='renewal rate that cycle',
            marker_color='#B2BEC3', opacity=0.6)
fig.update_layout(title='retention, monthly subscribers: one cliff then a plateau',
                  xaxis_title='billing cycle', yaxis_title='share', yaxis_tickformat='.0%')
fig.show()

ret_seg = []
for s in ORDER:
    d = MON.index[MON['segment'].eq(s)]
    r = retention(d, min_n=60)
    r['segment'] = s
    ret_seg.append(r)
ret_seg = pd.concat(ret_seg, ignore_index=True)
fig = px.line(ret_seg, x='cycle', y='S', color='segment', markers=True,
              category_orders={'segment': ORDER}, title='survival by segment')
fig.update_yaxes(tickformat='.0%', title='share still paying')
fig.show()
print('\nsurvival by segment:')
print(ret_seg[ret_seg['cycle'].isin([1, 2, 3])].pivot(index='segment', columns='cycle', values='S').round(3).to_string())

# what the loose definition was doing: any subscription charge within 37 days counts as a renewal
old_ren = set(ch.loc[~ch['is_credits'] & ch['days'].gt(1) & ch['days'].le(PERIOD + 7), 'customer'])
cmp = pd.DataFrame({'segment': MON['segment'],
                    'any charge <=37d (loose)': MON.index.isin(old_ren),
                    'renewed cycle 1 (used)': alive.reindex(MON.index)[1].fillna(False)})
tab = cmp.groupby('segment')[['any charge <=37d (loose)', 'renewed cycle 1 (used)']].mean().reindex(ORDER)
tab['inflation_pp'] = (tab.iloc[:, 0] - tab.iloc[:, 1]) * 100
print('\nfirst renewal under the loose 37-day window vs the definition used here:')
print(tab.round(3).to_string())
print('-> the loose window credits mid-cycle upgrades and win-backs as renewals, and it flatters the '
      'heaviest segments most, because they are the ones that upgrade mid-cycle.')

recovered revenue set aside: $91,954 from 1,321 accounts (4.3% of all revenue)
accounts that miss a cycle and pay again later: 512 of 19,530 (2.6%) -> treating the first miss as the end of the lifetime is safe; it is not quietly counting dunning gaps as churn
monthly, observable: 17,138
 cycle  n_at_risk  renewal    S
     0      17138     1.00 1.00
     1      17138     0.51 0.51
     2       6007     0.69 0.35
     3       2173     0.79 0.28
     4       1116     0.81 0.22
     5        329     0.80 0.18



survival by segment:
cycle              1    2    3
segment                       
1. one-shot     0.48 0.34 0.28
2. weak week-1  0.51 0.34 0.27
3. habit, light 0.49 0.33 0.26
4. habit, heavy 0.55 0.39 0.31
5. top usage    0.59 0.39  NaN

first renewal under the loose 37-day window vs the definition used here:
                 any charge <=37d (loose)  renewed cycle 1 (used)  inflation_pp
segment                                                                        
1. one-shot                          0.46                    0.48         -2.00
2. weak week-1                       0.52                    0.51          0.58
3. habit, light                      0.50                    0.49          0.69
4. habit, heavy                      0.62                    0.55          6.64
5. top usage                         0.77                    0.59         17.40
-> the loose window credits mid-cycle upgrades and win-backs as renewals, and it flatters the heaviest segments most, because t

In [28]:
# the dashboard version, next to the same customers tracked month over month
cal_sub = sub.copy()
cal_sub['ym'] = cal_sub['revenue_time'].dt.tz_localize(None).dt.to_period('M').astype(str)
sets = cal_sub.groupby('ym')['customer'].apply(set)
months = list(sets.index)
logo = pd.DataFrame([{'month': b,
                      'naive logo churn': 1 - len(sets[b]) / len(sets[a]),
                      'same-customer churn': 1 - len(sets[a] & sets[b]) / len(sets[a])}
                     for a, b in zip(months, months[1:])])
print(logo.round(3).to_string(index=False))

fig = go.Figure()
fig.add_scatter(x=logo['month'], y=logo['naive logo churn'], name='1 - A(t+1)/A(t)',
                line=dict(dash='dash', color='#B2BEC3'), mode='lines+markers')
fig.add_scatter(x=logo['month'], y=logo['same-customer churn'], name='same customers, paid again?',
                line=dict(color='#6C5CE7'), mode='lines+markers')
fig.add_hline(y=0, line_dash='dot')
fig.update_layout(title='churn two ways: the naive version tracks acquisition',
                  yaxis_tickformat='.0%', xaxis_title='month', yaxis_title='churn')
fig.update_xaxes(type='category')  # months are labels, not a continuous date axis
fig.show()

  month  naive logo churn  same-customer churn
2025-05             -2.06                 0.47
2025-06             -0.20                 0.51
2025-07             -1.04                 0.43
2025-08             -0.29                 0.47
2025-09              0.40                 0.44


### Insight — one cliff, then a plateau, and a definition that was flattering the heavy segments

**49% churn at the first renewal, then 19–31% per cycle.** This is not a steady leak; it is one cliff followed by an ordinary subscription business. Anything that moves the first bill is worth several times the same effort spent later.

**The loose definition was not neutral.** Counting any charge within 37 days as a renewal overstates the first renewal by **17pp for top usage and 7pp for heavy**, and by ~0 for everyone else — because those are the segments that upgrade mid-cycle, and an upgrade is not a renewal. "Heavy users retain better" survives, but the true gap is 48% → 59%, not 46% → 77%.

## 1.3 Paid churn is not usage churn

In [29]:
gen_alive = {int(m): set(g['customer']) for m, g in gen[gen['cycle'] >= 0].groupby('cycle')}
rows = []
for m in range(0, 7):
    elig = MON[MON['tenure_days'] >= PERIOD * (m + 1) - TOL]
    if len(elig) < 500:
        break
    payers = elig.index[alive.reindex(elig.index)[m].fillna(False)]
    used = payers.isin(list(gen_alive.get(m, set())))
    rev_m = piv.reindex(payers)[m].fillna(0)
    rows.append({'cycle': m, 'n_eligible': len(elig), 'still paying': len(payers) / len(elig),
                 'of payers, generated': used.mean(), 'dormant payers': (~used).sum(),
                 'their $ this cycle': rev_m[~used].sum(),
                 'share of cycle $': rev_m[~used].sum() / rev_m.sum()})
pu = pd.DataFrame(rows)
print(pu.round(3).to_string(index=False))

fig = px.line(pu.melt(id_vars='cycle', value_vars=['still paying', 'of payers, generated'],
                      var_name='metric', value_name='share'),
              x='cycle', y='share', color='metric', markers=True,
              title='of the accounts still paying, how many still generate')
fig.update_yaxes(tickformat='.0%', title='share')
fig.show()

 cycle  n_eligible  still paying  of payers, generated  dormant payers  their $ this cycle  share of cycle $
     0       17138          1.00                  0.99             183            5,195.39              0.01
     1       12531          0.51                  0.46            3395           89,121.07              0.36
     2        6539          0.36                  0.44            1319           35,393.51              0.36
     3        4102          0.29                  0.40             729           17,640.40              0.34
     4        1272          0.29                  0.52             181            4,923.74              0.20


### Insight — a third of the revenue is dormant

Of the accounts that renewed once, only **46% generated anything in the cycle they had just paid for** (99% did in cycle 0). Those dormant payers are **\$89,121 — 36% of cycle-1 revenue** — and the share stays near a third through cycle 3.

Two readings, both actionable. It is an **early-warning list available now**: usage stops before payment does, so this cycle's dormant payers are next cycle's churn. And it is a **liability**: revenue from people who forgot they subscribed returns as refunds and bad reviews, while sitting inside GRR looking like healthy retention.

## 1.4 What churn costs

$$\text{GRR}_m = \frac{B_m - \text{churned} - \text{contraction}}{B_m}, \qquad \text{NRR}_m = \text{GRR}_m + \frac{\text{expansion}}{B_m}$$

$B_m$ is what cycle-*m* payers paid in cycle *m*. On billing cycles, not calendar months, so renewals do not smear across boundaries. Credits count as expansion — a top-up is the same dollar as an upgrade.

In [30]:
ten_m = MON['tenure_days']
rows = []
for m in sorted(c for c in piv.columns if c + 1 in piv.columns):
    obs = ten_m[ten_m >= PERIOD * (m + 1) + TOL].index      # cycle m+1 must be decided
    cur = piv.loc[piv.index.isin(obs), m].fillna(0)
    nxt_ = piv.loc[piv.index.isin(obs), m + 1].fillna(0)
    cur, nxt_ = cur[cur > 0], nxt_[cur > 0]
    if len(cur) < 500:      # the tail cycles are too thin to read
        continue
    b, churned = cur.sum(), cur[nxt_ == 0].sum()
    delta = (nxt_ - cur)[nxt_ > 0]
    contraction, expansion = -delta[delta < 0].sum(), delta[delta > 0].sum()
    rows.append({'cycle': f'{m}->{m+1}', 'n': len(cur), 'base_$': b, 'churned_$': churned,
                 'contraction_$': contraction, 'expansion_$': expansion,
                 'GRR': (b - churned - contraction) / b,
                 'NRR': (b - churned - contraction + expansion) / b})
rr = pd.DataFrame(rows)
print(rr.round(3).to_string(index=False))

fig = go.Figure()
fig.add_bar(x=rr['cycle'], y=rr['GRR'], name='GRR', marker_color='#E17055')
fig.add_bar(x=rr['cycle'], y=rr['NRR'], name='NRR', marker_color='#00B894')
fig.add_hline(y=1.0, line_dash='dot')
fig.update_layout(title='revenue retention between billing cycles', barmode='group',
                  yaxis_tickformat='.0%', xaxis_title='cycle transition', yaxis_title='retention')
fig.show()

fig = px.bar(rr.melt(id_vars='cycle', value_vars=['churned_$', 'contraction_$', 'expansion_$'],
                     var_name='movement', value_name='usd'),
             x='cycle', y='usd', color='movement', barmode='relative',
             color_discrete_map={'churned_$': '#E17055', 'contraction_$': '#FDCB6E', 'expansion_$': '#00B894'},
             title='where the revenue moves, $ per cycle transition')
fig.update_yaxes(title='$')
fig.show()

cycle     n     base_$  churned_$  contraction_$  expansion_$  GRR  NRR
 0->1 17138 626,790.69 283,043.43      53,664.07    70,053.02 0.46 0.57
 1->2  6127 241,331.77  79,105.99      17,443.61    23,447.98 0.60 0.70
 2->3  2377  98,092.49  22,496.79       8,828.35    16,024.47 0.68 0.84
 3->4  1321  58,241.32  15,719.60       5,970.89    12,954.72 0.63 0.85


In [31]:
# the first-renewal leak, by segment
d = pd.DataFrame({'rev_0': piv.reindex(MON.index)[0].fillna(0),
                  'renewed': alive.reindex(MON.index)[1].fillna(False).astype(int),
                  'segment': MON['segment']})
d = d[d['rev_0'] > 0]
imp = d.groupby('segment').agg(n=('rev_0', 'size'), churn_rate=('renewed', lambda s: 1 - s.mean()),
                               rev_c0=('rev_0', 'sum'))
imp['lost'] = d[d['renewed'].eq(0)].groupby('segment')['rev_0'].sum()
imp['share_of_lost_$'] = imp['lost'] / imp['lost'].sum()
imp['share_of_base_$'] = imp['rev_c0'] / imp['rev_c0'].sum()
imp = imp.reindex([s for s in ORDER if s in imp.index])
print(imp.round(3).to_string())
print(f'\ncycle-0 revenue: ${imp["rev_c0"].sum():,.0f} | not renewed: ${imp["lost"].sum():,.0f} '
      f'({imp["lost"].sum() / imp["rev_c0"].sum():.1%})')

fig = px.bar(imp.reset_index().melt(id_vars='segment', value_vars=['share_of_base_$', 'share_of_lost_$'],
                                    var_name='share of', value_name='v'),
             x='segment', y='v', color='share of', barmode='group', category_orders={'segment': ORDER},
             title='share of cycle-0 revenue vs share of revenue lost at the first renewal')
fig.update_yaxes(tickformat='.0%', title='share')
fig.show()

                    n  churn_rate     rev_c0      lost  share_of_lost_$  share_of_base_$
segment                                                                                 
1. one-shot      5149        0.52 112,808.22 57,172.85             0.19             0.18
2. weak week-1   6149        0.49 193,134.15 92,234.40             0.30             0.31
3. habit, light  3307        0.51 113,754.34 56,504.86             0.19             0.18
4. habit, heavy  2079        0.45 146,770.58 69,578.66             0.23             0.23
5. top usage      454        0.41  60,323.40 27,143.89             0.09             0.10

cycle-0 revenue: $626,791 | not renewed: $302,635 (48.3%)


### Insight — one bill is the whole problem, and no segment is to blame for it

**GRR 46% and NRR 57% at the first transition; 60–68% and 70–85% at every one after.** Survivors expand, but not enough to cross 100% once win-backs are excluded — so expansion softens churn here, it does not out-run it.

**\$302,635 — 48% of first-cycle revenue — does not renew**, and it leaks almost exactly in proportion to what each segment contributes:

| segment | churn | share of cycle-0 \$ | share of lost \$ |
|---|---|---|---|
| 1. one-shot | 52% | 18% | 19% |
| 2. weak week-1 | 49% | 31% | 30% |
| 3. habit, light | 51% | 18% | 19% |
| 4. habit, heavy | 45% | 23% | 23% |
| 5. top usage | 41% | 10% | 9% |

Churn only spans 52% → 41% across the whole segmentation, so **there is no segment to single out — the first bill is the problem, not a particular kind of customer.** That is a change from the loose definition, under which the heavy segments looked far stickier than they are.

---
# 2. LTV

## 2.1 Gross LTV

$$\text{LTV}_{gross} = \sum_{m=0}^{H} \frac{r_m \, S(m)}{(1+d)^m}$$

- $r_m$ — **observed** revenue per surviving customer in cycle $m$ (subscription + credits, so expansion is inside LTV).
- $S(m)$ — the chained renewal curve from §1.2. No survival model: the renewal rates are read straight off the data, and the at-risk denominator does the censoring work.
- $d$ — monthly discount from 20% a year; $H = 36$ cycles.

**The extrapolation rule decides the answer**, because five cycles of history cannot speak for thirty-six:

- a cycle is only read off the data while it still holds **5% of the group** (min 50 accounts);
- past that, revenue is held flat and survival decays at the **mean renewal rate of cycles 2+** — excluding cycle 1, which is a much worse regime.

`ARPU / churn` is shown for contrast: one churn rate for everybody, taken from the worst cycle, applied forever, undiscounted.

In [32]:
D_ANNUAL, HORIZON = 0.20, 36
D = (1 + D_ANNUAL) ** (1 / 12) - 1
t = np.arange(HORIZON + 1)
disc = (1 + D) ** t

def per_survivor(customers):
    # observed revenue and job counts per paying survivor, by cycle
    idx = pd.Index(customers)
    ten_ = base['tenure_days'].reindex(idx)
    r = rev_cycle[rev_cycle['customer'].isin(idx)]
    gg = gen[(gen['cycle'] >= 0) & gen['customer'].isin(idx)]
    rows = []
    for m in range(0, 8):
        done = set(ten_[ten_ >= PERIOD * (m + 1) - TOL].index)
        live = done & set(alive.index[alive[m]]) if m in alive.columns else set()
        if len(live) < 15:
            break
        jm = gg[gg['customer'].isin(live) & gg['cycle'].eq(m)]
        rows.append({'cycle': m, 'n_alive': len(live),
                     'r_m': r.loc[r['customer'].isin(live) & r['cycle'].eq(m), 'rev'].sum() / len(live),
                     'g_img': (len(jm) - jm['is_video'].sum()) / len(live),
                     'g_vid': jm['is_video'].sum() / len(live)})
    return pd.DataFrame(rows)

def floor_n(n):
    return max(50, int(0.05 * n))

def schedule(obs, col, horizon=HORIZON):
    ok = obs[obs['n_alive'] >= floor_n(obs['n_alive'].iloc[0])]
    ok = ok if len(ok) else obs.iloc[:1]
    s, m_last = ok.set_index('cycle')[col], int(ok['cycle'].max())
    tail = float(s.iloc[-2:].mean()) if len(s) >= 2 else float(s.iloc[-1])
    return np.array([float(s[m]) if m <= m_last else tail for m in range(horizon + 1)])

def s_curve(ret, horizon=HORIZON, r_long=None):
    ok = ret[ret['n_at_risk'] >= floor_n(ret['n_at_risk'].iloc[0])]
    ok = ok if len(ok) > 1 else ret.iloc[:2]
    s, m_last = ok.set_index('cycle')['S'], int(ok['cycle'].max())
    if r_long is None:
        post = ok.loc[ok['cycle'] >= 2, 'renewal']
        r_long = float(post.mean()) if len(post) else float(ok['renewal'].iloc[-1])
    vals = [1.0]
    for m in range(1, horizon + 1):
        vals.append(float(s[m]) if m <= m_last else vals[-1] * r_long)
    return np.array(vals), r_long, m_last

R_OBS = per_survivor(MON.index)
print('observed revenue and jobs per surviving customer:')
print(R_OBS.round(2).to_string(index=False))

r_vec = schedule(R_OBS, 'r_m')
S_vec, R_LONG, M_LAST = s_curve(RET)
ltv = float((r_vec * S_vec / disc).sum())
ltv_nodisc = float((r_vec * S_vec).sum())
ltv_flat = float((r_vec * np.where(t <= M_LAST, S_vec, S_vec[M_LAST]) / disc).sum())
arpu_w = float((R_OBS['r_m'] * R_OBS['n_alive']).sum() / R_OBS['n_alive'].sum())
theta = 1 - RET.set_index('cycle')['renewal'][1]

print(f'\nread from data to cycle {M_LAST}; beyond it survival decays {1 - R_LONG:.1%} per cycle '
      f'(mean renewal of cycles 2-{M_LAST}), revenue held flat at ${r_vec[-1]:.2f}')
print(f'S: {np.round(S_vec[:M_LAST + 1], 3)} -> S(12)={S_vec[12]:.3f}, S(36)={S_vec[36]:.3f}')
print(f'\nARPU ${arpu_w:.2f} | cycle-1 churn {theta:.1%}')
print(f'LTV, ARPU / churn                      = ${arpu_w / theta:,.0f}')
print(f'LTV, retention x revenue, no discount  = ${ltv_nodisc:,.0f}')
print(f'LTV, retention x revenue, discounted   = ${ltv:,.0f}   <- chosen')
print(f'   [contrast] survival held flat after cycle {M_LAST}: ${ltv_flat:,.0f} ({ltv_flat / ltv:.1f}x)')

fig = px.bar(pd.DataFrame({'method': ['ARPU / churn', 'no discount', f'{D_ANNUAL:.0%} discount, {HORIZON} cycles (chosen)'],
                           'ltv': [arpu_w / theta, ltv_nodisc, ltv]}),
             x='method', y='ltv', text_auto='.0f', title='gross LTV by method')
fig.update_layout(yaxis_title='$ per customer', xaxis_title='')
fig.show()

cum_pv = pd.DataFrame({'cycle': t, 'pv': r_vec * S_vec / disc})
cum_pv['cumulative'] = cum_pv['pv'].cumsum()
fig = px.area(cum_pv, x='cycle', y='cumulative', title='where gross LTV accumulates (present value)')
fig.update_yaxes(title='$ cumulative')
fig.show()
print(f'\ncycle 0 = {cum_pv["pv"].iloc[0] / ltv:.0%} of LTV | first 12 cycles = {cum_pv["cumulative"].iloc[12] / ltv:.0%}')

observed revenue and jobs per surviving customer:
 cycle  n_alive   r_m  g_img  g_vid
     0    17138 36.57  92.62  24.30
     1     6330 39.19  73.47  13.43
     2     2337 42.00  49.95  14.63
     3     1208 42.37  66.47  13.74
     4      375 66.67 196.97  24.51
     5       39 56.17 127.72  24.49

read from data to cycle 4; beyond it survival decays 23.8% per cycle (mean renewal of cycles 2-4), revenue held flat at $42.19
S: [1.    0.506 0.348 0.275 0.222] -> S(12)=0.025, S(36)=0.000

ARPU $38.34 | cycle-1 churn 49.4%
LTV, ARPU / churn                      = $78
LTV, retention x revenue, no discount  = $122
LTV, retention x revenue, discounted   = $117   <- chosen
   [contrast] survival held flat after cycle 4: $312 (2.7x)



cycle 0 = 31% of LTV | first 12 cycles = 98%


In [33]:
# by segment: own retention curve, own revenue schedule
rows = []
for s in ORDER:
    cs = MON.index[MON['segment'].eq(s)]
    obs, ret = per_survivor(cs), retention(cs, min_n=60)
    if len(obs) < 2 or len(ret) < 2:
        continue
    sv, r_l, m_l = s_curve(ret)
    rows.append({'segment': s, 'n': len(cs), 'r_0': obs['r_m'].iloc[0], 'S(1)': ret['S'].iloc[1],
                 'steady_churn': 1 - r_l, 'gross_ltv': float((schedule(obs, 'r_m') * sv / disc).sum())})
SEG_LTV = pd.DataFrame(rows)
SEG_LTV['x_vs_one_shot'] = SEG_LTV['gross_ltv'] / SEG_LTV['gross_ltv'].iloc[0]
print(SEG_LTV.round(3).to_string(index=False))

fig = px.bar(SEG_LTV, x='segment', y='gross_ltv', text_auto='.0f',
             category_orders={'segment': ORDER}, title='gross LTV by segment')
fig.update_layout(yaxis_title='$ per customer', xaxis_title='')
fig.show()

        segment    n    r_0  S(1)  steady_churn  gross_ltv  x_vs_one_shot
    1. one-shot 5149  21.91  0.48          0.23      72.33           1.00
 2. weak week-1 6149  31.41  0.51          0.24     104.36           1.44
3. habit, light 3307  34.40  0.49          0.26     109.45           1.51
4. habit, heavy 2079  70.60  0.55          0.26     232.66           3.22
   5. top usage  454 132.87  0.59          0.34     334.46           4.62


In [34]:
# what actually moves the answer
rows = []
for da in [0.10, 0.20, 0.30, 0.40]:
    for hz in [12, 24, 36, 60]:
        sv, _, _ = s_curve(RET, hz)
        rows.append({'discount': f'{da:.0%}', 'horizon': hz,
                     'ltv': float((schedule(R_OBS, 'r_m', hz) * sv / ((1 + (1 + da) ** (1 / 12) - 1) ** np.arange(hz + 1))).sum())})
sens = pd.DataFrame(rows).pivot(index='discount', columns='horizon', values='ltv')
fig = px.imshow(sens, text_auto='.0f', aspect='auto', color_continuous_scale='Purples',
                title='gross LTV: discount rate x horizon')
fig.update_xaxes(title='horizon, cycles')
fig.show()

tail = pd.DataFrame([{'steady-state churn': f'{c:.1%}',
                      'ltv': float((r_vec * s_curve(RET, r_long=1 - c)[0] / disc).sum())}
                     for c in sorted([0.05, 0.08, 1 - R_LONG, 0.15, 0.20, 0.25])])
fig = px.bar(tail, x='steady-state churn', y='ltv', text_auto='.0f',
             title='gross LTV vs assumed churn beyond the observed window')
fig.update_layout(yaxis_title='$ per customer')
fig.show()
print(f'discount x horizon: ${sens.values.min():,.0f} - ${sens.values.max():,.0f} '
      f'({sens.values.max() / sens.values.min():.1f}x)')
print(f'tail-churn assumption alone: ${tail["ltv"].min():,.0f} - ${tail["ltv"].max():,.0f} '
      f'({tail["ltv"].max() / tail["ltv"].min():.1f}x)  <- the assumption that matters')

discount x horizon: $111 - $119 (1.1x)
tail-churn assumption alone: $115 - $203 (1.8x)  <- the assumption that matters


### Insight — \$117, bracketed by two easy mistakes

| method | LTV | what it gets wrong |
|---|---|---|
| `ARPU / churn` | \$78 | one churn rate for everybody, taken from the worst cycle, forever, undiscounted |
| retention × revenue, flat tail | \$312 | assumes the last observed cycle repeats — survival never decays |
| **retention × revenue, decaying tail, discounted** | **\$117** | chosen |

**98% of it lands within 12 cycles**, so a 12-month payback rule loses almost nothing — which is the right horizon to promise on six months of data anyway.

**Segments span 4.6×**: \$72 / \$104 / \$109 / \$233 / \$334. Revenue does the work (\$21.91 → \$132.87 in cycle 0) more than retention does (0.48 → 0.59), so a single blended CAC target overpays for one-shot and underbids for top usage.

**The tail assumption is the only argument worth having**: discount rate and horizon together move LTV 1.1× (\$111–119), the post-window churn rate moves it 1.8× (\$115–203).

## 2.2 Net LTV: the cost ceiling

Net LTV is gross LTV minus fees, minus compute, minus CAC. Fees are knowable (~3.5%) and the **job counts are in the data**, but cost per job and CAC are not — so a net LTV number would be an assumption dressed as a result.

One useful thing can still be computed exactly: the compute cost at which a segment stops paying for itself.

$$c^{*}_{img} = \frac{\sum_m r_m(1-f)S(m)/(1+d)^m}{\sum_m \left(g^{img}_m + k\,g^{vid}_m\right)S(m)/(1+d)^m}, \qquad k = \frac{c_{vid}}{c_{img}}$$

Above $c^{*}$ the segment is sold at a loss before a cent of CAC. Two missing numbers become one question finance can answer in a minute.

In [35]:
F_FEE, K_VIDEO = 0.035, 8.0   # payment+FX fees; assumed cost ratio of a video job to an image job

def cost_ceiling(obs, ret, k=K_VIDEO, f=F_FEE):
    r_, gi, gv = (schedule(obs, c) for c in ['r_m', 'g_img', 'g_vid'])
    sv, _, _ = s_curve(ret)
    gross_pv = float((r_ * (1 - f) * sv / disc).sum())
    pv_img, pv_vid = float((gi * sv / disc).sum()), float((gv * sv / disc).sum())
    return gross_pv, pv_img, pv_vid, gross_pv / (pv_img + k * pv_vid)

rows = []
for s in ORDER:
    cs = MON.index[MON['segment'].eq(s)]
    obs, ret = per_survivor(cs), retention(cs, min_n=60)
    if len(obs) < 2 or len(ret) < 2:
        continue
    g_pv, pi, pv, cstar = cost_ceiling(obs, ret)
    rows.append({'segment': s, 'gross_pv_after_fees': g_pv, 'img_jobs_pv': pi, 'video_jobs_pv': pv,
                 'cost_weighted_jobs': pi + K_VIDEO * pv, 'c_star_img': cstar, 'c_star_vid': cstar * K_VIDEO})
BE = pd.DataFrame(rows)
print(BE.round(4).to_string(index=False))
print(f'\nwhole monthly base: ${cost_ceiling(R_OBS, RET)[3]:.4f} per image job')

fig = px.bar(BE, x='segment', y='c_star_img', text_auto='.3f', category_orders={'segment': ORDER},
             title=f'breakeven cost per image job (video = {K_VIDEO:.0f}x), before CAC')
fig.update_layout(yaxis_title='$ per image job', xaxis_title='')
fig.show()

        segment  gross_pv_after_fees  img_jobs_pv  video_jobs_pv  cost_weighted_jobs  c_star_img  c_star_vid
    1. one-shot                69.79        25.56          10.40              108.73        0.64        5.13
 2. weak week-1               100.71       116.37          36.96              412.05        0.24        1.96
3. habit, light               105.62        78.16          48.99              470.11        0.22        1.80
4. habit, heavy               224.52       571.35         170.35            1,934.14        0.12        0.93
   5. top usage               322.76     4,787.83         349.24            7,581.78        0.04        0.34

whole monthly base: $0.1807 per image job


### Insight — the cost ceiling runs backwards to LTV

| segment | gross LTV | breakeven \$/image job |
|---|---|---|
| 1. one-shot | \$72 | **\$0.64** |
| 2. weak week-1 | \$104 | \$0.24 |
| 3. habit, light | \$109 | \$0.22 |
| 4. habit, heavy | \$233 | \$0.12 |
| 5. top usage | \$334 | **\$0.04** |

**A 16× spread, in the reverse order of LTV.** Top usage carries ~70× the cost-weighted job load of one-shot for 4.6× the revenue, so its margin per dollar is the thinnest in the base. If the true cost per image is near \$0.01 the ranking holds and top usage is the best customer Higgsfield has; above ~\$0.04 it is sold at a loss while one-shot stays comfortably profitable.

**So the ranking of the segments is only as reliable as one number nobody has asked for yet: cost per generation.** That question is worth more than any refinement to this model. After it, CAC by channel.

---
# 3. History — how the segments moved

Aggregate value falling has two causes with opposite fixes: **mix shift** (acquisition brings different people) or **within-segment decay** (the same people are worth less). The aggregate number cannot tell them apart, so decompose it symmetrically:

$$\Delta = \underbrace{\sum_s (w_s^1 - w_s^0)\frac{v_s^0 + v_s^1}{2}}_{\text{mix}} + \underbrace{\sum_s \frac{w_s^0 + w_s^1}{2}(v_s^1 - v_s^0)}_{\text{value}}$$

Cohort = month of first charge. September contributes no cohort (§0.2), and April is the launch month, so both endpoints get checked.

In [36]:
size = SEG.groupby('cohort').size()
keep = [c for c in sorted(SEG['cohort'].unique()) if size[c] >= 300]
print('cohort sizes:'); print(size.to_string())
print(f'\nAugust is now ~complete: {size[keep[-1]]:,} of {(base["cohort"].eq(keep[-1])).sum():,} accounts '
      f'that first paid that month (first payment must be on or before '
      f'{(DATA_END - pd.Timedelta(days=OBS_MIN)).date()})')

mix = pd.crosstab(SEG['cohort'], SEG['segment'], normalize='index')[ORDER].loc[keep]
print('\nsegment mix by cohort:'); print(mix.round(3).to_string())
fig = px.area(mix.reset_index().melt(id_vars='cohort', var_name='segment', value_name='share'),
              x='cohort', y='share', color='segment', category_orders={'segment': ORDER},
              title='segment mix by cohort')
fig.update_yaxes(tickformat='.0%', title='share of cohort')
fig.update_xaxes(type='category')  # cohorts are labels, not a continuous date axis
fig.show()

fig = px.bar(SEG.groupby(['cohort', 'period_label']).size().rename('n').reset_index(),
             x='cohort', y='n', color='period_label', barmode='stack', title='cohort size and billing mix')
fig.update_yaxes(title='accounts')
fig.update_xaxes(type='category')
fig.show()

cohort sizes:
cohort
2025-04    1181
2025-05    2989
2025-06    2569
2025-07    6272
2025-08    6114

August is now ~complete: 6,114 of 6,519 accounts that first paid that month (first payment must be on or before 2025-08-29)

segment mix by cohort:
segment  1. one-shot  2. weak week-1  3. habit, light  4. habit, heavy  5. top usage
cohort                                                                              
2025-04         0.20            0.37             0.31             0.11          0.00
2025-05         0.33            0.36             0.26             0.05          0.00
2025-06         0.26            0.40             0.23             0.10          0.01
2025-07         0.33            0.36             0.15             0.13          0.03
2025-08         0.30            0.34             0.16             0.16          0.04


In [37]:
# within-segment value and retention by cohort
val = (SEG[SEG['cohort'].isin(keep)].groupby(['cohort', 'segment'])
       .agg(n=('arpu_meq', 'size'), arpu_meq=('arpu_meq', 'mean'),
            m1_jobs=('m1_jobs', 'median'), jobs_per_usd=('jobs_per_usd', 'median')).reset_index())
val = val[val['n'] >= 40]
for col, title in [('arpu_meq', 'ARPU (monthly-equivalent, first 30 days)'),
                   ('jobs_per_usd', 'median jobs per $ (margin proxy)')]:
    fig = px.line(val, x='cohort', y=col, color='segment', markers=True,
                  category_orders={'segment': ORDER}, title=f'{title} by cohort')
    fig.update_xaxes(type='category')
    fig.show()
print('ARPU_meq by cohort x segment:')
print(val.pivot(index='cohort', columns='segment', values='arpu_meq').round(2).to_string())

MON['renewed'] = alive.reindex(MON.index)[1].fillna(False).astype(int)
ren = MON[MON['cohort'].isin(keep)].groupby(['cohort', 'segment'])['renewed'].agg(['size', 'mean'])
print('\nfirst-renewal rate by cohort x segment (monthly):')
print(ren[ren['size'] >= 40]['mean'].unstack().round(3).to_string())

ARPU_meq by cohort x segment:
segment  1. one-shot  2. weak week-1  3. habit, light  4. habit, heavy  5. top usage
cohort                                                                              
2025-04        22.95           32.82            38.34           135.72           NaN
2025-05        20.24           27.53            35.72           121.02           NaN
2025-06        24.86           31.03            33.27            74.69           NaN
2025-07        22.07           29.97            33.88            52.50        140.01
2025-08        21.28           34.77            32.04            69.78        131.21

first-renewal rate by cohort x segment (monthly):
segment  1. one-shot  2. weak week-1  3. habit, light  4. habit, heavy  5. top usage
cohort                                                                              
2025-04         0.54            0.56             0.60             0.64           NaN
2025-05         0.51            0.50             0.49             0.5

In [38]:
MIN_CELL = 40
cnt = pd.crosstab(SEG['cohort'], SEG['segment'])[ORDER]
w_raw = pd.crosstab(SEG['cohort'], SEG['segment'], normalize='index')[ORDER]
v = SEG.groupby(['cohort', 'segment'])['arpu_meq'].mean().unstack()[ORDER]

def decompose(c0, c1, plot=False, detail=False):
    segs = [s for s in ORDER if min(cnt.loc[c0, s], cnt.loc[c1, s]) >= MIN_CELL]
    w = w_raw[segs].div(w_raw[segs].sum(axis=1), axis=0)
    w0, w1 = w.loc[c0], w.loc[c1]
    v0, v1 = v.loc[c0, segs], v.loc[c1, segs]
    tot0, tot1 = float((w0 * v0).sum()), float((w1 * v1).sum())
    mix_eff = float(((w1 - w0) * (v0 + v1) / 2).sum())
    val_eff = float((((w0 + w1) / 2) * (v1 - v0)).sum())
    print(f'\n{c0} -> {c1}: ARPU_meq ${tot0:.2f} -> ${tot1:.2f} (delta ${tot1 - tot0:+.2f})'
          f'{"   excluded, too few accounts: " + str([s for s in ORDER if s not in segs]) if len(segs) < len(ORDER) else ""}')
    print(f'  mix   (who we acquire)      ${mix_eff:+.2f}')
    print(f'  value (what they are worth) ${val_eff:+.2f}')
    if detail:
        det = pd.DataFrame({'n_first': cnt.loc[c0, segs], 'n_last': cnt.loc[c1, segs],
                            'w_first': w0, 'w_last': w1, 'v_first': v0, 'v_last': v1})
        det['mix_contrib'] = (w1 - w0) * (v0 + v1) / 2
        det['value_contrib'] = ((w0 + w1) / 2) * (v1 - v0)
        print(det.round(3).to_string())
    if plot:
        fig = go.Figure(go.Waterfall(x=[c0, 'mix', 'value', c1], y=[tot0, mix_eff, val_eff, tot1],
                                     measure=['absolute', 'relative', 'relative', 'total'],
                                     decreasing=dict(marker=dict(color='#E17055')),
                                     increasing=dict(marker=dict(color='#00B894')),
                                     totals=dict(marker=dict(color='#6C5CE7'))))
        fig.update_layout(title=f'ARPU change {c0} -> {c1}: mix vs value', yaxis_title='$ per customer')
        fig.update_xaxes(type='category')  # '2025-04' would otherwise be read as a date
        fig.show()

decompose(keep[0], keep[-1], plot=True, detail=True)
print('\n--- April is the launch month and the smallest cohort, so redo it from May ---')
decompose(keep[1], keep[-1], detail=True)


2025-04 -> 2025-08: ARPU_meq $43.99 -> $35.90 (delta $-8.09)   excluded, too few accounts: ['5. top usage']
  mix   (who we acquire)      $+2.28
  value (what they are worth) $-10.36
                 n_first  n_last  w_first  w_last  v_first  v_last  mix_contrib  value_contrib
segment                                                                                       
1. one-shot          242    1818     0.21    0.31    22.95   21.28         2.33          -0.43
2. weak week-1       433    2078     0.37    0.36    32.82   34.77        -0.42           0.71
3. habit, light      370     978     0.32    0.17    38.34   32.04        -5.18          -1.52
4. habit, heavy      131     965     0.11    0.17   135.72   69.78         5.54          -9.12



--- April is the launch month and the smallest cohort, so redo it from May ---

2025-05 -> 2025-08: ARPU_meq $32.17 -> $35.90 (delta $+3.72)   excluded, too few accounts: ['5. top usage']
  mix   (who we acquire)      $+7.15
  value (what they are worth) $-3.43
                 n_first  n_last  w_first  w_last  v_first  v_last  mix_contrib  value_contrib
segment                                                                                       
1. one-shot          980    1818     0.33    0.31    20.24   21.28        -0.34           0.33
2. weak week-1      1087    2078     0.36    0.36    27.53   34.77        -0.25           2.60
3. habit, light      763     978     0.26    0.17    35.72   32.04        -2.98          -0.78
4. habit, heavy      158     965     0.05    0.17   121.02   69.78        10.72          -5.59


In [39]:
# is retention itself moving?
rows = []
for c in keep:
    d = MON[MON['cohort'].eq(c)]
    if len(d) < 100:
        continue
    p, n = d['renewed'].mean(), len(d)
    rows.append({'cohort': c, 'n': n, 'first_renewal': p, 'ci95': 1.96 * np.sqrt(p * (1 - p) / n)})
rc = pd.DataFrame(rows)
print(rc.round(3).to_string(index=False))
fig = go.Figure(go.Scatter(x=rc['cohort'], y=rc['first_renewal'], mode='lines+markers',
                           error_y=dict(array=rc['ci95']), line=dict(color='#6C5CE7')))
fig.update_layout(title='first-renewal rate by cohort (monthly, 95% CI)',
                  yaxis_tickformat='.0%', yaxis_title='renewed once', xaxis_title='cohort')
fig.update_xaxes(type='category')
fig.show()

 cohort    n  first_renewal  ci95
2025-04 1051           0.58  0.03
2025-05 2709           0.50  0.02
2025-06 2351           0.51  0.02
2025-07 5635           0.52  0.01
2025-08 5392           0.48  0.01


### Insight — pick the wrong baseline and you get the wrong strategy

| window | ΔARPU | mix | value |
|---|---|---|---|
| Apr → Aug | **−\$8.09** | +\$2.28 | **−\$10.36** |
| May → Aug | **+\$3.72** | **+\$7.15** | −\$3.43 |

From April this is a business whose customers are getting cheaper; from May it is one acquiring better customers faster than their value erodes. April is the launch month (1,181 accounts, 131 of them heavy), so **the honest statement is that ARPU has been roughly flat since May.**

**What both windows agree on:** heavy users are arriving in far greater numbers and each is worth less — share 5% → 17% while their 30-day ARPU falls \$121 → \$70. Habit-light is shrinking (26% → 17%), top usage is growing at a stable ~\$131–140.

Two explanations, opposite fixes: heavy users are self-selecting onto cheaper plans (packaging), or wider acquisition is reaching a less committed audience (channel). The \$70-vs-\$121 gap is about one plan tier, which favours packaging — **plan mix by cohort is the query that settles it.**

**Retention is drifting too**: first renewal 58% → 48% (±1–3pp). So the \$117 LTV is a backward-looking average; on August's traffic it would come out lower, and that is the number to use for CAC decisions today.

---
# 4. Summary

**Churn.** No cancellation events exist, so churn is a missing charge: no scheduled subscription charge in a fully-elapsed 30-day cycle. No grace period — the renewal distribution has no gap for one to sit in (the rebill cluster is 30–32 days, and what follows is a flat trickle of win-backs at a median of 64 days). Win-backs are recovered revenue, not retention, worth 3.3pp on the first renewal; mid-cycle upgrades are not renewals either, worth up to 17pp on the heaviest segments. Annual is excluded because 87% never pay twice and none reach day 365. The last month has no new subscriptions, which makes it a pure retention month and the reason every cohort through August is measurable. Result: **49% churn at the first bill, then 19–31% per cycle**, segment survival 0.48 → 0.59.

**Revenue impact.** GRR 46% / NRR 57% at the first transition, 60–68% / 70–85% after. **\$302,635 (48% of first-cycle revenue) never renews**, in proportion to each segment's revenue — the first bill is the problem, not a segment. Separately, dormant payers (paying, generating nothing) are next month's churn, identifiable now.

**Gross LTV.** $\sum_m r_m S(m)/(1+d)^m$ — observed revenue per survivor × chained renewal rates, 20% annual discount, 36 cycles, revenue held flat and survival decayed at the post-cliff rate past the last well-populated cycle. **\$117 per monthly customer** (\$115–203 across tail assumptions), against \$78 from `ARPU / churn` and \$312 with a flat tail. Per segment: **\$72 / \$104 / \$109 / \$233 / \$334**.

**Net LTV.** Fees are knowable and job counts are in the data, but cost per job and CAC are not, so the useful output is the **cost ceiling**: \$0.64 per image job for one-shot down to \$0.04 for top usage. Above ~\$0.04 the segment ranking inverts. Cost per generation is the most valuable number missing from this dataset.

**History.** Mix vs value flips sign depending on whether April or May is the baseline, so: ARPU flat since May, heavy users growing as a share (5% → 17%) at falling value (\$121 → \$70), habit-light shrinking, first renewal drifting 58% → 48%.